# Experiments 46 - 47
Impact of applying exGreen techniques to create a derived dataset.

- **Model:**
    1. `yolov8m` *(Medium)*
- **Dataset:** 3.5m | 90º
    1. exGreen masked _(gray images)_
    2. 2 PCA + exGreen _(false color images)_
- **Crop Tiles:** 640px
- **Sizes:** small & mid
- **Experiments:**
    1. exGreen
    2. 2 PCA + exGreen

## Init

In [1]:
import os
import shutil
import fnmatch
import pickle
import torch

In [2]:
!pip install ultralytics

### Disabling augmentation

In [3]:
# IF default augmentation is not desiered, use the following line
# !pip uninstall albumentations

    # Disable all type of augmentation
    augment=False,
    erasing = 0,
    hsv_h=0,
    hsv_s=0,
    hsv_v=0,
    degrees=0.0,
    translate=0,
    scale=0.5,
    shear=0.0,
    flipud=0.0,
    fliplr=0.0

## Helper Functions

In [4]:
# Change for different file formats
reference = {
  "small": {
    "suffix": ".S",
    "file": "209"
  },
  "mid": {
    "suffix": ".M",
    "file": "503"
  },
  "large": {
    "suffix": ".L",
    "file": "000"
  }
}

In [5]:
# Clone config files
def copy_config(src_folder, dest_folder):
    """
    Copies files from src_folder to dest_folder, excluding subfolders.

    Args:
        src_folder: The path to the source folder.
        dest_folder: The path to the destination folder.
    """

    try:
        # Ensure destination folder exists
        os.makedirs(dest_folder, exist_ok=True)

        for filename in os.listdir(src_folder):
            src_path = os.path.join(src_folder, filename)
            dest_path = os.path.join(dest_folder, filename)

            if os.path.isfile(src_path):
                shutil.copy2(src_path, dest_path) #copy metadata as well.
                #Use shutil.copy for not copying metadata.
                print(f"Copied: {filename}")
            #else: #optional
                #print(f"Skipped (not a file): {filename}") #optional. Uncomment if you want to see the skipped folders.

        print("✅ Copying complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")


In [6]:
# Copy filtered dataset images/labels
def copy_and_filter_folder(src_folder, dest_folder, pattern):
    """
    Copies a folder and files that match the given pattern.
    Alerts the user when a folder or file already exists but *does not* overwrite.
    Creates only what is needed.

    :param src_folder: Path to the source folder.
    :param dest_folder: Path to the destination folder.
    :param pattern: Filename pattern to keep (e.g., "*.txt").
    """
    try:
        # Ensure destination folder exists
        if not os.path.exists(dest_folder):
            print(f"✓ Creating destination folder '{dest_folder}'.\n")
            os.makedirs(dest_folder)
        else:
            print(f"✓ Destination folder '{dest_folder}' already exists.\n")

        # Walk through the source folder
        for root, _, files in os.walk(src_folder):
            relative_path = os.path.relpath(root, src_folder)
            new_root = os.path.join(dest_folder, relative_path)

            if not os.path.exists(new_root):
                print(f"Creating subdirectory '{new_root}'")
                os.makedirs(new_root)
            else:
                print(f"❕Subdirectory '{new_root}' already exists.")
                print("Make sure the data inside is relevant. Otherwise, just delete the folder and repeat the cloning process.")

            for file in files:
                if fnmatch.fnmatch(file, pattern + "*"):
                    src_file = os.path.join(root, file)
                    dest_file = os.path.join(new_root, file)

                    if not os.path.exists(dest_file):
                        shutil.copy2(src_file, dest_file)  # copy metadata as well
                    else:
                        print(f"❗️File '{dest_file}' already exists. Skipping.")

            print(f" ✓ Copying files complete.\n")
        print("✅ Copying dataset complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")

In [7]:
def copy_directory(source, destination, overwrite=True):
    try:
        if overwrite and os.path.exists(destination):
            shutil.rmtree(destination)  # Remove existing directory
        shutil.copytree(source, destination)
        print(f"Directory '{source}' copied to '{destination}'")
        return True  # Indicate success
    except FileExistsError:
        print(f"Destination '{destination}' exists. Use overwrite=True to replace.")
        return False  # Indicate failure

In [8]:
# Load/Download prediction results (BB + confidence) as .pkl file

def save_results(results, filename):
    with open(filename, 'wb') as f:
        pickle.dump(results, f)

def load_results(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)

In [9]:
def save_on_cloud(source: str, destination: str):
    """
    Saves a folder to a cloud storage location (e.g., Google Drive in Colab).

    Args:
        source (str): The path to the source folder.
        destination (str): The path to the destination folder (in cloud storage).
    """
    # 0. Input Validation (Assertions)
    assert isinstance(source, str), "Source must be a string."
    assert isinstance(destination, str), "Destination must be a string."

    try:
        # 1. Verify Source Folder
        if not os.path.exists(destination):
            os.makedirs(destination)

        # 2. Copy the Folder
        shutil.copytree(source, destination, dirs_exist_ok=True)
        print("✅ Folder copied successfully:\n  ",source,"\n  -->",destination)

    except Exception as e:
        print(f"❌ An error occurred: {e}")

# Datasets builder

## Importing from Drive

In [1]:
!rm -rf /content/sample_data

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# Check if the cloud path is ok and the dataset can be found
!ls /content/drive/MyDrive/YOLO

3.5m.v3i.yolov8.640px		       3.5m.v3i.yolov8_pca.640px
3.5m.v3i.yolov8.640px.aug.v1	       best_e26.pt
3.5m.v3i.yolov8.640px.aug.v1.soil_aug  Inference
3.5m.v3i.yolov8.640px.soil_aug	       models
3.5m.v3i.yolov8_blended.640px	       optuna_yolov8_f1_study.db
3.5m.v3i.yolov8_exgreen.640px	       runs
3.5m.v3i.yolov8_masked.640px	       Untitled


In [14]:
drive_path = '/content/drive/MyDrive/YOLO'
drive_datasets_paths = os.listdir(drive_path)
drive_datasets = len(drive_datasets_paths)
if (drive_datasets) > 1:
    print("There are %d dataset options:" % drive_datasets)
else:
    print("Theres is only 1 dataset:")
drive_datasets_paths

There are 14 dataset options:


['3.5m.v3i.yolov8.640px',
 'Inference',
 'models',
 'runs',
 '3.5m.v3i.yolov8.640px.aug.v1',
 'best_e26.pt',
 'optuna_yolov8_f1_study.db',
 '3.5m.v3i.yolov8.640px.soil_aug',
 '3.5m.v3i.yolov8.640px.aug.v1.soil_aug',
 '3.5m.v3i.yolov8_blended.640px',
 '3.5m.v3i.yolov8_masked.640px',
 '3.5m.v3i.yolov8_exgreen.640px',
 '3.5m.v3i.yolov8_pca.640px',
 'Untitled']

In [33]:
choose_dataset = 12
index = choose_dataset - 1
model_name = os.listdir(drive_path)[index]
print("Chosen model:", model_name)

Chosen model: 3.5m.v3i.yolov8_exgreen.640px


***Readme:***
*   **Option 1:** is desirable if you need to test many subset combinations in the same session (avoid downloading data twice from the cloud).
*   **Option 2:** is desired if you are goint to work just with one dataset  (avoid downloading unnecessary data from the cloud).
*   **Option 3:** is best if you're just going to test one subset combination  (avoid downloading any data from the cloud).



In [34]:
# Option 2 (download just the needed dataset)
cloud_path = f"/content/drive/MyDrive/YOLO/{model_name}/"
local_path = f"/content/YOLO/"
!mkdir $local_path
!cp -r $cloud_path $local_path

mkdir: cannot create directory ‘/content/YOLO/’: File exists


In [35]:
src_folder = f"/content/YOLO/{model_name}"
data = f"{src_folder}/data.yaml"

## Download model

In [36]:
from ultralytics import YOLO

In [37]:
# Load pretrain YOLO v8 model
model = YOLO("yolov8m.pt")

In [38]:
# BEST MODEL: Load stored model (Exp. 26)
# model = YOLO("/content/drive/MyDrive/YOLO/best_e26.pt")

# Finetuning

### Optimization

In [39]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [43]:
# Garbage collection
import gc
torch.cuda.empty_cache()
gc.collect()

0

In [41]:
# Reduce VRAM usage by reducing fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

### Info

In [20]:
!nvidia-smi

Tue Apr 29 13:34:13 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [21]:
!yolo version

8.3.120


-----
## Experiment 46
### *YOLOv8 Mid | exGreen images*
Images created by applying exGreen

### Train

In [44]:
# Set's maximum training time (in hours)
time: float = 3 # Depending on remaining Colab cuota
# This overrides the epochs argument, allowing training to automaticallystop after the specified duration.
# Useful for time-constrained training scenarios. (aka Colab cuotas)

In [45]:
# Train model
history = model.train(
    data=data,
    epochs=500,
    imgsz=640,
    batch=-1,
    freeze=10,
    patience=200,
    #time = time,
)

Ultralytics 8.3.120 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov8m.pt, data=/content/YOLO/3.5m.v3i.yolov8_exgreen.640px/data.yaml, epochs=500, time=None, patience=200, batch=-1, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=10, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, sh

train: Scanning /content/YOLO/3.5m.v3i.yolov8_exgreen.640px/train/labels... 216 images, 0 backgrounds, 0 corrupt: 100%|██████████| 216/216 [00:00<00:00, 2081.53it/s]

train: New cache created: /content/YOLO/3.5m.v3i.yolov8_exgreen.640px/train/labels.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (Tesla T4) 14.74G total, 0.24G reserved, 0.23G allocated, 14.26G free


      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output
    25856899       79.07         1.153          28.8         19.91        (1, 3, 640, 640)                    list
    25856899       158.1         1.365         34.58         32.02        (2, 3, 640, 640)                    list
    25856899       316.3         1.751         60.79         52.25        (4, 3, 640, 640)                    list
    25856899       632.5         2.519         77.58         58.93        (8, 3, 640, 640)                    list
    25856899        1265         3.846         143.4         110.3       (16, 3, 640, 640)                    list
AutoBatch: Using batch-size 44 for CUDA:0 9.37G/14.74G (64%) ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1735.3±447.0 MB/s, size: 78.3 KB)


train: Scanning /content/YOLO/3.5m.v3i.yolov8_exgreen.640px/train/labels.cache... 216 images, 0 backgrounds, 0 corrupt: 100%|██████████| 216/216 [00:00<?, ?it/s]

...

    242/500      7.32G      1.137     0.6215      1.007       1355        640: 100%|██████████| 5/5 [00:04<00:00,  1.16it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:01<00:00,  1.29it/s]

                   all        108       2409      0.481      0.383      0.349     0.0977
EarlyStopping: Training stopped early as no improvement observed in last 200 epochs. Best results observed at epoch 42, best model saved as best.pt.
To update EarlyStopping(patience=200) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



242 epochs completed in 0.463 hours.
Optimizer stripped from runs/detect/train/weights/last.pt, 52.1MB
Optimizer stripped from runs/detect/train/weights/best.pt, 52.1MB

Validating runs/detect/train/weights/best.pt...
Ultralytics 8.3.120 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:02<00:00,  1.34s/it]


                   all        108       2409      0.469      0.412      0.367       0.11
Speed: 0.2ms preprocess, 10.9ms inference, 0.0ms loss, 2.8ms postprocess per image
Results saved to runs/detect/train


In [46]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a5b7c3657d0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [47]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v3i.yolov8_exgreen.640px/data.yaml',
          epochs=500,
          time=None,
          patience=200,
          batch=44,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=10,
          multi_scale=False,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.0,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
          iou=

In [48]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train


### Validation

In [49]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [50]:
# Load stored model
# model = YOLO("/content/drive/MyDrive/save/detect/train/weights/best.pt")

In [51]:
# Validate the model
results = model.val(data=data)

Ultralytics 8.3.120 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1779.6±410.1 MB/s, size: 73.7 KB)


val: Scanning /content/YOLO/3.5m.v3i.yolov8_exgreen.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:08<00:00,  1.21s/it]


                   all        108       2409      0.466      0.417      0.367       0.11
Speed: 6.7ms preprocess, 22.0ms inference, 0.0ms loss, 5.5ms postprocess per image
Results saved to runs/detect/val


In [52]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val


### Save results

In [53]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/save1/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/save1/


-----
## Experiment 47
### *YOLOv8 Mid | False color images*
False color images are created by applying:
1. Excess Green to a grayscale image.
1. A 2-component PCA (to reduce dimensionality).
1. Combining these images as RGB channels.

### Train

In [ ]:
# Set's maximum training time (in hours)
time: float = 3 # Depending on remaining Colab cuota
# This overrides the epochs argument, allowing training to automaticallystop after the specified duration.
# Useful for time-constrained training scenarios. (aka Colab cuotas)

In [24]:
# Train model
history = model.train(
    data=data,
    epochs=500,
    imgsz=640,
    batch=-1,
    freeze=10,
    patience=300,
    #time = time,
)

Ultralytics 8.3.120 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov8m.pt, data=/content/YOLO/3.5m.v3i.yolov8_pca.640px/data.yaml, epochs=500, time=None, patience=300, batch=-1, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train2, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=10, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_

train: Scanning /content/YOLO/3.5m.v3i.yolov8_pca.640px/train/labels... 216 images, 0 backgrounds, 0 corrupt: 100%|██████████| 216/216 [00:00<00:00, 469.56it/s]

train: New cache created: /content/YOLO/3.5m.v3i.yolov8_pca.640px/train/labels.cache


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (Tesla T4) 14.74G total, 0.24G reserved, 0.23G allocated, 14.26G free
      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output
    25856899       79.07         1.153         42.55         173.1        (1, 3, 640, 640)                    list
    25856899       158.1         1.365         34.54         70.51        (2, 3, 640, 640)                    list
    25856899       316.3         1.730         60.73         79.88        (4, 3, 640, 640)                    list
    25856899       632.5         2.498         75.47         72.98        (8, 3, 640, 640)                    list
    25856899        1265         3.

train: Scanning /content/YOLO/3.5m.v3i.yolov8_pca.640px/train/labels.cache... 216 images, 0 backgrounds, 0 corrupt: 100%|██████████| 216/216 [00:00<?, ?it/s]

...

    340/500      7.25G      1.236     0.6902       1.08         37        640: 100%|██████████| 6/6 [00:04<00:00,  1.20it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:02<00:00,  1.02s/it]

                   all        108       2409      0.473      0.472      0.407      0.124
EarlyStopping: Training stopped early as no improvement observed in last 300 epochs. Best results observed at epoch 40, best model saved as best.pt.
To update EarlyStopping(patience=300) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



340 epochs completed in 0.672 hours.
Optimizer stripped from runs/detect/train2/weights/last.pt, 52.1MB
Optimizer stripped from runs/detect/train2/weights/best.pt, 52.1MB

Validating runs/detect/train2/weights/best.pt...
Ultralytics 8.3.120 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:02<00:00,  1.18s/it]


                   all        108       2409      0.515      0.473      0.447      0.148
Speed: 0.2ms preprocess, 10.7ms inference, 0.0ms loss, 2.4ms postprocess per image
Results saved to runs/detect/train2


In [25]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a5b79d19790>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [26]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v3i.yolov8_pca.640px/data.yaml',
          epochs=500,
          time=None,
          patience=300,
          batch=43,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train2',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=10,
          multi_scale=False,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.0,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
          iou=0.7

In [27]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train2


### Validation

In [28]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [29]:
# Load stored model
# model = YOLO("/content/drive/MyDrive/save/detect/train/weights/best.pt")

In [30]:
# Validate the model
results = model.val(data=data)

Ultralytics 8.3.120 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2849.6±815.9 MB/s, size: 116.6 KB)


val: Scanning /content/YOLO/3.5m.v3i.yolov8_pca.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:09<00:00,  1.35s/it]


                   all        108       2409      0.509      0.474      0.446      0.148
Speed: 3.4ms preprocess, 24.9ms inference, 0.0ms loss, 4.4ms postprocess per image
Results saved to runs/detect/val


In [31]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val


### Save results

In [32]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/save2/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/save2/
